# Stellarator injection

A neutral beam in a 3D equilibrium from [GVEC](https://github.com/gvec-group/gvec): the same `deposit` and `slow_down` as in the tokamak examples, on a `GVECEquilibrium` instead of an axisymmetric `Equilibrium`. Both satisfy `MagneticEquilibrium`, the protocol the beam codes use: `rho_at`, `field_at`, `inside_at` at Cartesian points, a bounding box and the flux shells.

GVEC is an optional extra: `pip install freeofcharge[gvec]`. DESC equilibria work the same way through `DESCEquilibrium` (`pip install freeofcharge[desc]`), for example `DESCEquilibrium.from_example("W7-X")`.

## A small stellarator

A rotating ellipse with two field periods, solved by GVEC in a few seconds at low resolution. A W7-X equilibrium is loaded the same way, from its parameter and state files with `GVECEquilibrium.from_files`.

In [ ]:
import tempfile

import gvec
import matplotlib.pyplot as plt
import numpy as np
import plasma_plots as pp
import pyvista as pv

import freeofcharge as foc
from freeofcharge.gvec_equilibrium import GVECEquilibrium

parameters = {
    "ProjectName": "ellipse",
    "nfp": 2,
    "which_hmap": 1,
    "PhiEdge": 0.5,
    "X1X2_deg": 3,
    "LA_deg": 3,
    "sgrid": {"grid_type": 0, "nElems": 3},
    "X1_mn_max": (2, 1),
    "X2_mn_max": (2, 1),
    "LA_mn_max": (2, 1),
    "X1_sin_cos": "_cos_",
    "X2_sin_cos": "_sin_",
    "LA_sin_cos": "_sin_",
    # the boundary: R = 3 + 0.4 cos(theta) + 0.05 cos(theta - 2 zeta), Z likewise
    "X1_b_cos": {(0, 0): 3.0, (1, 0): 0.4, (1, 1): 0.05},
    "X2_b_sin": {(1, 0): 0.6, (1, 1): 0.05},
    "init_average_axis": True,
    "iota": {"type": "polynomial", "coefs": [0.4, 0.1]},
    "pres": {"type": "polynomial", "coefs": [0.0]},
    "minimize_tol": 1e-6,
    "totalIter": 2000,
    "logIter": 1000,
}
run = gvec.run(parameters, runpath=tempfile.mkdtemp(), quiet=True)
equilibrium = GVECEquilibrium(run.state)
print(isinstance(equilibrium, foc.MagneticEquilibrium), equilibrium.nfp)

The flux surfaces turn with the toroidal angle. [plasma-plots](https://max-models.github.io/plasma-plots/) reads GVEC's evaluations (`pp.from_gvec`) and draws them: `|B|` on a poloidal plane with the flux-coordinate lines, and on the surface `rho = 0.5` over one field period. `rho` is the square root of the normalised toroidal flux here, not of the poloidal flux as in the axisymmetric class.

In [ ]:
ev = pp.from_gvec(
    run.state.evaluate("mod_B", "pos", "N_FP", rho=17, theta=65, zeta=[0.0, np.pi / 2])
)
ev.mod_B.attrs["units"] = "T"
ev.mod_B.plasma.plot.slice(
    coords="physical",
    plane="RZ",
    zeta=0.0,
    overlays={"coordinate_lines": {"rho": 4, "theta": 8}},
)
plt.show()

In [ ]:
surface = pp.from_gvec(
    run.state.evaluate("mod_B", "N_FP", rho=[0.5], theta=64, zeta=64)
)
surface.mod_B.attrs["units"] = "T"
surface.mod_B.plasma.plot.slice(x="zeta", y="theta", rho=0.5)
plt.show()

## Flux shells

`flux_surfaces` returns the same `FluxSurfaces` as in 2D: shell volumes, toroidally averaged cross sections, `|B|` with the flux-surface averaging weights (the Jacobian) on a `(theta, zeta)` grid, the rotational transform and the trapped fraction. This toy has a strong mirror along the axis (`|B|` from 0.53 to 0.83 T already at `rho = 0.1`), hence the large trapped fraction even near the axis.

In [ ]:
rho_edges = np.linspace(0.0, 1.0, 11)
surfaces = equilibrium.flux_surfaces(rho_edges)
total = float(gvec.evaluate(run.state, "V", rho=None, theta=None, zeta=None)["V"])
print(f"volume {surfaces.volume.sum():.4f} m^3, GVEC {total:.4f} m^3")

fig, ax = plt.subplots(figsize=(6, 3.5), layout="constrained")
ax.plot(surfaces.rho, surfaces.rotational_transform, "o-", label="iota")
ax.plot(surfaces.rho, surfaces.trapped_fraction, "s-", label="trapped fraction")
ax.set(xlabel="rho", ylim=(0, 1))
ax.legend()
plt.show()

## Deposition

A hydrogen beam at 55 keV, the energy of W7-X's first NBI experiments, aimed tangentially at `R = 2.9 m`. The lines are straight in Cartesian space; at every step `rho_at` inverts GVEC's map (a Newton iteration per point, about 30 µs each), so keep the markers and steps modest. Shine-through falls steeply with the density, as in the tokamak.

In [ ]:
origin, direction = foc.tangential_injection(2.9, 5.0, toroidal_angle=0.3)
beam = foc.NeutralBeam.gaussian(
    origin,
    direction,
    55e3,
    1.8e6,
    mass_number=1,
    width=0.1,
    divergence=0.01,
    markers=200,
    seed=1,
)
ions = {"H": (1, 1, 1.0)}
results = {}
for ne0 in (2e19, 6e19):
    plasma = foc.Plasma.parabolic(ne0, 3e3, ions=ions)
    results[ne0] = foc.deposit(
        beam,
        equilibrium,
        plasma,
        step=0.01,
        rho_edges=rho_edges,
        births_per_marker=1,
        seed=2,
    )
    print(f"n_e0 = {ne0:.0e}: shine-through {results[ne0].shine_through_fraction:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.8), layout="constrained")
for ne0, color in zip(results, ("C0", "C1"), strict=True):
    ax.step(
        surfaces.rho,
        results[ne0].power_density(surfaces) / 1e3,
        where="mid",
        color=color,
        label=f"{ne0:.0e}",
    )
ax.set(xlabel="rho", ylabel="deposited power [kW/m^3]")
ax.legend(title="n_e0 [m^-3]")
plt.show()

In 3D: the last closed surface over the whole torus, coloured by `|B|` and translucent, with every tenth beam line and the births at the lower density. Click the picture for the interactive scene (vtk.js) and turn it.

In [ ]:
torus = pp.from_gvec(
    run.state.evaluate(
        "mod_B",
        "pos",
        "N_FP",
        rho=[1.0],
        theta=48,
        zeta=np.linspace(0.0, 2 * np.pi, 64 * equilibrium.nfp, endpoint=False),
    )
)
torus.mod_B.attrs["units"] = "T"
plotter = torus.mod_B.plasma.plot.slices_3d(show_domain=False, opacity=0.35)
starts = np.asarray(beam.markers.positions)
directions = np.asarray(beam.directions)
for k in range(0, len(starts), 10):
    plotter.add_mesh(
        pv.Line(starts[k], starts[k] + 7.0 * directions[k]), color="black", line_width=1
    )
plotter.add_points(
    np.asarray(results[2e19].births.positions),
    color="orange",
    point_size=5,
    render_points_as_spheres=True,
)
plotter.camera_position = "iso"
pp.scene(
    plotter,
    alt="A translucent stellarator surface coloured by |B|, beam lines and the births",
    title="Beam births in the stellarator",
)

## Slowing down

The local slowing down works per shell, so it carries over unchanged: heating of electrons and ions and the fast-ion density. The orbit average (`orbit_average=True`) uses the axisymmetric bounce averages of `freeofcharge.trapping` and is switched off here; helically trapped ions and 3D orbits are the next steps. The current drive is shown in the tokamak example only, since its trapped-electron correction assumes axisymmetry.

In [ ]:
plasma = foc.Plasma.parabolic(6e19, 3e3, ions=ions)
slowing = foc.slow_down(results[6e19], plasma, surfaces, orbit_average=False)
data = slowing.to_dataset()
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 3.8), layout="constrained")
for species in data.species.values:
    ax0.step(
        data.rho, data.heating.sel(species=species) / 1e3, where="mid", label=species
    )
ax0.set(xlabel="rho", ylabel="heating [kW/m^3]")
ax0.legend()
ax1.step(data.rho, data.fast_ion_density, where="mid")
ax1.set(xlabel="rho", ylabel="fast-ion density [m^-3]")
plt.show()